# Combining Dense Vector Search with Graph Context

While GraphRAG provides exceptional structural multi-hop precision, and vector search provides broad semantic conceptual matching, relying on either approach exclusively leaves gaps:

Pure Vector Search misses relational boundaries and structural entity connections.

Pure Graph Search misses unstructured nuance, narrative context, and descriptive paragraphs that don't neatly map to discrete triples.

This module covers Hybrid Graph-Vector Retrieval, which combines both modalities into a unified context pipeline.


## 1. Architectural Pattern: How Graph and Vector Fusion Works

The hybrid graph-vector architecture merges retrieval paths in parallel:

The Vector Stream: Takes the raw user query, embeds it, and fetches the top-$K$ most semantically relevant text chunks from the vector database.

The Graph Stream: Identifies key entities in the user query, traverses the Knowledge Graph (using NetworkX or Neo4j) to extract structural relational paths, and converts those paths into readable summaries or text fragments.

The Fusion & Reranking Layer: Combines both sets of results (using Reciprocal Rank Fusion or context window concatenation) to construct a comprehensive context block for the LLM.

## 2. Implementation Code (03_hybrid_graph_vector.py)
This standalone Python script simulates a hybrid retrieval engine combining local vector similarity (via cosine distance math or a mock search) with graph traversal paths.

In [ ]:
"""
03_hybrid_graph_vector.py
Demonstrates combining dense vector chunk retrieval with structured 
Knowledge Graph context traversal for a hybrid GraphRAG pipeline.
"""

import networkx as nx
from typing import List, Dict, Any

class HybridGraphVectorRAG:
    def __init__(self):
        # 1. Initialize In-Memory Knowledge Graph (NetworkX)
        self.knowledge_graph = nx.DiGraph()
        self._populate_mock_graph()

        # 2. Initialize Mock Vector Database Store (Simulating text chunks)
        self.vector_store = [
            {
                "chunk_id": "chunk_101",
                "content": "TechCorp Europe experienced major supply chain bottlenecks in Q3 due to port strikes in Berlin.",
                "embedding_tags": ["supply chain", "Europe", "TechCorp", "Berlin"]
            },
            {
                "chunk_id": "chunk_102",
                "content": "DataStream Logistics provides tier-1 routing solutions for automated distribution networks across Germany.",
                "embedding_tags": ["DataStream", "logistics", "Germany"]
            }
        ]

    def _populate_mock_graph(self):
        """Populates the local knowledge graph with structural enterprise entities."""
        triples = [
            ("TechCorp Global", "OWNS", "TechCorp Europe"),
            ("TechCorp Europe", "LOCATED_IN", "Berlin"),
            ("TechCorp Europe", "PARTNERS_WITH", "DataStream Logistics"),
            ("DataStream Logistics", "OPERATES_IN", "Germany")
        ]
        for subj, rel, obj in triples:
            self.knowledge_graph.add_edge(subj, obj, relation=rel)

    def search_vector_store(self, query: str) -> List[Dict[str, Any]]:
        """Simulates dense vector retrieval by checking keyword overlap in tags."""
        query_terms = set(query.lower().split())
        results = []
        for chunk in self.vector_store:
            matches = sum(1 for tag in chunk["embedding_tags"] if tag.lower() in query_terms)
            if matches > 0:
                results.append({"source": "vector_store", "id": chunk["chunk_id"], "content": chunk["content"], "score": matches})
        return results

    def search_graph_context(self, entity_name: str) -> List[str]:
        """Traverses the knowledge graph to extract structural relationships around an entity."""
        graph_contexts = []
        if entity_name in self.knowledge_graph:
            # Outgoing relations (1-hop)
            for neighbor in self.knowledge_graph.successors(entity_name):
                rel = self.knowledge_graph.get_edge_data(entity_name, neighbor)["relation"]
                graph_contexts.append(f"Graph Fact: {entity_name} -[{rel}]-> {neighbor}")
            
            # Incoming relations (1-hop)
            for predecessor in self.knowledge_graph.predecessors(entity_name):
                rel = self.knowledge_graph.get_edge_data(predecessor, entity_name)["relation"]
                graph_contexts.append(f"Graph Fact: {predecessor} -[{rel}]-> {entity_name}")
                
        return graph_contexts

    def hybrid_retrieve(self, query: str, target_entity: str) -> Dict[str, Any]:
        """Executes both vector search and graph traversal, merging the findings."""
        print(f"Executing Hybrid Retrieval for Query: '{query}' (Target Entity: {target_entity})")
        
        # Stream 1: Dense Vector Search
        vector_results = self.search_vector_store(query)
        
        # Stream 2: Structured Graph Traversal
        graph_results = self.search_graph_context(target_entity)
        
        return {
            "vector_chunks": vector_results,
            "graph_paths": graph_results
        }

# --- Execution Block ---
if __name__ == "__main__":
    hybrid_rag = HybridGraphVectorRAG()
    
    # User query requiring both semantic chunk details and structural graph paths
    user_query = "What caused supply chain issues for TechCorp Europe and who are their logistics partners?"
    entity_focus = "TechCorp Europe"
    
    retrieved_data = hybrid_rag.hybrid_retrieve(user_query, entity_focus)
    
    print("\n--- Fused Hybrid Context Payload ---")
    print("\n[Vector Search Stream (Unstructured Text Chunks)]")
    for chunk in retrieved_data["vector_chunks"]:
        print(f"- ID: {chunk['id']} | Content: {chunk['content']}")
        
    print("\n[Graph Search Stream (Structured Relational Facts)]")
    for fact in retrieved_data["graph_paths"]:
        print(f"- {fact}")

## 3. Production Design Considerations
Context Window Management: Fusing full text chunks with extensive graph subgraphs can rapidly bloat your token count. Use strict filtering or summarization on graph paths before concatenating them with vector documents.

Asynchronous Execution: Because vector stores and graph databases (like Neo4j) are typically hosted on independent servers or clusters, execute both retrieval streams asynchronously using asyncio to prevent compounding query latency.